In [12]:
from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime


In [13]:
# For test purposes

csvFile = ["MSFT", "JNJ", "NVDA","GOOG"]

# A function that grabs the sectors of all the stocks
# and displays them into a data frame
def pull_stock_sectors (tickers):
    sectors = []

    for ticker in tickers:
        stock = yf.Ticker(ticker)
        sector = stock.info.get("sector")

        sectors.append({
            "Ticker": ticker,
            "Sector" :sector
        })
    return pd.DataFrame(sectors)


sector_data = pull_stock_sectors(csvFile)
sector_data

,Ticker,Sector
0,MSFT,Technology
1,JNJ,Healthcare
2,NVDA,Technology
3,GOOG,Communication Services


For this next part, the way I want to choose which sectors are best for each situation is through a scoring system. Where we rank each sector 1 to 5, with 1 being safe and 5 being risky. This will make it a lot easier later on to know which sectors we want to invest in for our portfolio.

In [ ]:
# Assigned scores for each sector
sector_scores = {
    "Utilities": 1,
    "Consumer Defensive": 1,
    "Healthcare": 2,
    "Real Estate": 2,
    "Financial Services": 3,
    "Industrials": 3,
    "Energy": 4,
    "Basic Materials": 4,
    "Communication Services": 4,
    "Consumer Cyclical": 4,
    "Technology": 5
}

# add the sector scores to our data frame of all the sectors 

def score_stock_sectors(sector_data):
    scores = []

    for sector in sector_data["Sector"]:
        score = sector_scores.get(sector)
        scores.append(score)

    sector_data["Sector_Score"] = scores

    return sector_data

# Calculates the total portfolio weight in each sector
def calculate_sector_weights(portfolio):
    sector_weights = portfolio.groupby("Sector")["Weight"].sum()
    return sector_weights


# Checks if any sector exceeds the 40% limit we have
def check_sector_limit(portfolio):
    sector_weights = calculate_sector_weights(portfolio)
    return (sector_weights <= 0.40 + 0.0000000001).all()


Under this is just me testing with a random input to see if the code is doing what its supposed to be doing

In [15]:
#for testing purposes
test_tickers = ["MSFT", "JNJ", "PG", "NVDA", "JPM", "NEE"]

sector_data = pull_stock_sectors(test_tickers)

sector_data = score_stock_sectors(sector_data)

display(sector_data)

# Test 1: Technology exceeds 40%
sector_data["Weight"] = [0.15, 0.15, 0.15, 0.15, 0.15, 0.25]

display(sector_data)
display(calculate_sector_weights(sector_data))

print("Test 1:", check_sector_limit(sector_data))

# Test 2: All sectors are within 40%
sector_data["Weight"] = [0.15, 0.15, 0.15, 0.15, 0.15, 0.25]

# Change NVDA's weight to 0.10 and NEE's to 0.30
sector_data["Weight"] = [0.15, 0.15, 0.15, 0.10, 0.15, 0.30]

print("Test 2:", check_sector_limit(sector_data))

,Ticker,Sector,Sector_Score
0,MSFT,Technology,5
1,JNJ,Healthcare,2
2,PG,Consumer Defensive,1
3,NVDA,Technology,5
4,JPM,Financial Services,3
5,NEE,Utilities,1


,Ticker,Sector,Sector_Score,Weight
0,MSFT,Technology,5,0.15
1,JNJ,Healthcare,2,0.15
2,PG,Consumer Defensive,1,0.15
3,NVDA,Technology,5,0.15
4,JPM,Financial Services,3,0.15
5,NEE,Utilities,1,0.25


Sector
Consumer Defensive    0.15
Financial Services    0.15
Healthcare            0.15
Technology            0.30
Utilities             0.25
Name: Weight, dtype: float64

Test 1: True
Test 2: True
